# V13 Self-Play (Colab L4)

**Goal:** generate ~1000 V13 self-play games to anchor the steady-state distribution for the pillar4a distillation corpus. Bulk of V13's training signal comes from crisis mining (run separately on M5); this notebook contributes the steady-state coverage.

Player: **pillar3a (sharp_25_epoch_12) + value_head_pillar3a + q_weight=2.0** — current strongest known player. A/B verified on 100 seeds cap=25K: MCTS@100 mean **21,310** (P50 17,644, %<1000=1.0%, P95 cap-bound 51,612). +30% mean / +37% P50 / floor cut 86% vs old value_head_v12_v12targets (HISTORY 158).

Settings:
- 200 sims, q_weight=2.0, batch_size=8 (HISTORY lesson 115)
- max-turns 10000 (slight headroom over pillar3a P75 ~8K turns)
- temperature-moves 15 (standard exploration)
- Dirichlet α=0.3, weight=0.25 (defaults)

**Upload to `MyDrive/alphatrain/` before running:**
1. `colorlines_path_b.tar.gz` (built by `scripts/build_path_b_tarball.sh`)
2. `sharp_25_epoch_12.pt` (~136 MB, this is pillar3a)
3. `value_head_pillar3a.pt` (~38 KB; upload the local file `value_head_sharp25_ep12.pt` under this Drive name)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/sharp_25_epoch_12.pt /content/alphatrain/data/pillar3a.pt
!cp {DRIVE}/value_head_pillar3a.pt /content/alphatrain/data/value_head_pillar3a.pt
print(f'Model: {os.path.getsize("/content/alphatrain/data/pillar3a.pt")/1e6:.0f} MB')
print(f'Value head: {os.path.getsize("/content/alphatrain/data/value_head_pillar3a.pt")/1e3:.0f} KB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

In [ ]:
# === V13 Self-Play (anchor the steady-state distribution) ===
# Distribute by editing SEED_START / SEED_END if running multiple instances.
# Crisis mining (M5 local) uses 800000-808000 — keep these ranges disjoint.
SEED_START = 1300000
SEED_END = 1301000   # 1000 games
# ============================================================

SIMS = 200               # 200 = balanced; pillar3a's strong priors mean we need less than V12's 400
Q_WEIGHT = 2.0           # calibrated for value_head_pillar3a (HISTORY 158)
WORKERS = 24             # Colab CPU
BATCH_SIZE = 8           # HISTORY lesson 115 — bs=8 is essential
MAX_TURNS = 10000        # pillar3a P95 ~34K policy / >50K MCTS; cap=10K = ~20K-pt headroom
TEMP_MOVES = 15          # standard exploration window
SAVE_DIR = f'{DRIVE}/selfplay_v13'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games)')
print(f'Sims: {SIMS}, q_weight: {Q_WEIGHT}, Cap: {MAX_TURNS} turns')
print(f'Workers: {WORKERS}, bs: {BATCH_SIZE}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/pillar3a.pt \
    --value-head-path /content/alphatrain/data/value_head_pillar3a.pt \
    --q-weight {Q_WEIGHT} \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves {TEMP_MOVES} \
    --max-turns {MAX_TURNS} \
    --compile

## Notes

- **Resume support:** if Colab disconnects, just re-run the previous cell. It skips seeds whose game files already exist in `SAVE_DIR`.
- **Wall time on L4:** ~30-60s per game in 24 workers parallel → ~12-20h for 1000 games (pillar3a at MCTS@200 is slightly slower per turn than V12's pillar2y2 because games last longer). May exceed Colab's 24h kill on bigger games — split SEED_END to 1300500 (500 games) for a safer 6-10h run, then a second notebook run for 1300500-1301000.
- **Watch:** GPU server prints `[GPU] X evals, Y fwd (avg bs=Z, ZZZZ evals/s)` every 10K forwards. avg bs should be ≥50 and evals/s in the thousands.
- **Output:** each game saved as `game_seed{N}_score{S}.json` in the save dir.
- **Crisis mining runs locally on M5 in parallel** (~12-25h depending on seed count). Once both are done, build_expert_v2_tensor.py merges selfplay/ + crisis/ JSONs into `v13_pillar3a.pt`.

## After V13 lands

Train pillar4a on M5 or Colab G4 (~12h, 17 epochs):

```
python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v13_pillar3a.pt --amp --compile \
    --resume alphatrain/data/pillar3a.pt --warm-start \
    --epochs 17 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --target-temperature 0.5    # or 0.7 if V13 entropy check shows sharper targets
```

Decision criterion: pillar4a ≥ +30% over pillar3a mean.